# Credit Card Feature Engineering

This notebook builds applicant-level features from the cleaned monthly credit-card data. It creates monthly ratios and flags, aggregates them up to one row per card account, then to one row per applicant, and fills in zeros for applicants with no credit-card history at all.


## Import libraries


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 150)
print("Libraries imported successfully.")

Libraries imported successfully.


## Set project paths


In [5]:
current_folder = Path.cwd().resolve()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder
input_path = project_root / "data" / "interim" / "credit_card_balance_clean.pkl"
application_path = project_root / "data" / "interim" / "application_clean.pkl"
training_ids_path = project_root / "data" / "modeling" / "splits" / "training_ids.csv"
test_ids_path = project_root / "data" / "modeling" / "splits" / "test_ids.csv"
output_path = project_root / "data" / "features" / "credit_card_features.pkl"
audit_folder = project_root / "reports" / "audits"
output_path.parent.mkdir(parents=True, exist_ok=True)
audit_folder.mkdir(parents=True, exist_ok=True)
for required_path in [input_path, application_path, training_ids_path, test_ids_path]:
    assert required_path.exists(), f"Required file was not found: {required_path}"
print("Clean input:", input_path)
print("Feature output:", output_path)

Clean input: /Users/taranveersingh/A-MRP/data/interim/credit_card_balance_clean.pkl
Feature output: /Users/taranveersingh/A-MRP/data/features/credit_card_features.pkl


## Load cleaned data and split information


In [7]:
credit = pd.read_pickle(input_path)
application_target = pd.read_pickle(application_path)[["SK_ID_CURR", "TARGET"]]
training_ids = pd.read_csv(training_ids_path)["SK_ID_CURR"]
test_ids = pd.read_csv(test_ids_path)["SK_ID_CURR"]
training_id_set = set(training_ids)
test_id_set = set(test_ids)
project_ids = training_id_set.union(test_id_set)
print("Clean monthly credit-card shape:", credit.shape)
print("Credit-card accounts represented:", credit["SK_ID_PREV"].nunique())
print("Applicants with credit-card history:", credit["SK_ID_CURR"].nunique())

Clean monthly credit-card shape: (3227965, 30)
Credit-card accounts represented: 87452
Applicants with credit-card history: 86905


Only about 28% of applicants (86,905 of 307,511) have any credit-card history at all.


## Create monthly credit-card features


In [10]:
def safe_ratio(numerator, denominator):
    return (numerator / denominator.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)

credit["CC_UTILIZATION"] = safe_ratio(credit["AMT_BALANCE"], credit["AMT_CREDIT_LIMIT_ACTUAL"])
credit["CC_DRAWING_LIMIT_RATIO"] = safe_ratio(credit["AMT_DRAWINGS_CURRENT"], credit["AMT_CREDIT_LIMIT_ACTUAL"])
credit["CC_PAYMENT_BALANCE_RATIO"] = safe_ratio(credit["AMT_PAYMENT_TOTAL_CURRENT"], credit["AMT_BALANCE"])
credit["CC_PAYMENT_MINIMUM_RATIO"] = safe_ratio(credit["AMT_PAYMENT_CURRENT"], credit["AMT_INST_MIN_REGULARITY"])
credit["CC_ATM_DRAWING_RATE"] = safe_ratio(credit["AMT_DRAWINGS_ATM_CURRENT"], credit["AMT_DRAWINGS_CURRENT"])
credit["CC_POS_DRAWING_RATE"] = safe_ratio(credit["AMT_DRAWINGS_POS_CURRENT"], credit["AMT_DRAWINGS_CURRENT"])
credit["CC_IS_DELINQUENT"] = credit["SK_DPD"].gt(0).astype("int8")
credit["CC_IS_SEVERELY_DELINQUENT"] = credit["SK_DPD"].gt(30).astype("int8")
credit["CC_IS_ACTIVE"] = credit["NAME_CONTRACT_STATUS"].eq("Active").astype("int8")
credit["CC_PAYMENT_INFO_AVAILABLE"] = credit["AMT_PAYMENT_CURRENT"].notna().astype("int8")
drawing_columns = ["AMT_DRAWINGS_ATM_CURRENT", "AMT_DRAWINGS_OTHER_CURRENT", "AMT_DRAWINGS_POS_CURRENT"]
credit["CC_DRAWING_INFO_AVAILABLE"] = credit[drawing_columns].notna().all(axis=1).astype("int8")
credit["CC_RECENT_12M"] = credit["MONTHS_BALANCE"].ge(-12).astype("int8")
print("Monthly credit-card features created: 12")

Monthly credit-card features created: 12


These describe things like how much of the credit limit is being used, how drawings compare to the limit, and whether the account is overdue.


## Aggregate monthly history to credit-card account level


In [13]:
account_features = credit.groupby(["SK_ID_CURR", "SK_ID_PREV"]).agg(
    CC_MONTH_COUNT=("MONTHS_BALANCE", "count"),
    CC_OLDEST_MONTH=("MONTHS_BALANCE", "min"),
    CC_LATEST_MONTH=("MONTHS_BALANCE", "max"),
    CC_BALANCE_MEAN=("AMT_BALANCE", "mean"),
    CC_BALANCE_MAX=("AMT_BALANCE", "max"),
    CC_CREDIT_LIMIT_MEAN=("AMT_CREDIT_LIMIT_ACTUAL", "mean"),
    CC_CREDIT_LIMIT_MAX=("AMT_CREDIT_LIMIT_ACTUAL", "max"),
    CC_UTILIZATION_MEAN=("CC_UTILIZATION", "mean"),
    CC_UTILIZATION_MAX=("CC_UTILIZATION", "max"),
    CC_DRAWING_TOTAL=("AMT_DRAWINGS_CURRENT", "sum"),
    CC_DRAWING_MEAN=("AMT_DRAWINGS_CURRENT", "mean"),
    CC_ATM_DRAWING_TOTAL=("AMT_DRAWINGS_ATM_CURRENT", "sum"),
    CC_POS_DRAWING_TOTAL=("AMT_DRAWINGS_POS_CURRENT", "sum"),
    CC_PAYMENT_TOTAL=("AMT_PAYMENT_TOTAL_CURRENT", "sum"),
    CC_PAYMENT_CURRENT_MEAN=("AMT_PAYMENT_CURRENT", "mean"),
    CC_MINIMUM_PAYMENT_MEAN=("AMT_INST_MIN_REGULARITY", "mean"),
    CC_PAYMENT_BALANCE_RATIO_MEAN=("CC_PAYMENT_BALANCE_RATIO", "mean"),
    CC_PAYMENT_MINIMUM_RATIO_MEAN=("CC_PAYMENT_MINIMUM_RATIO", "mean"),
    CC_DELINQUENT_MONTH_COUNT=("CC_IS_DELINQUENT", "sum"),
    CC_DELINQUENT_MONTH_RATE=("CC_IS_DELINQUENT", "mean"),
    CC_SEVERE_DELINQUENT_MONTH_COUNT=("CC_IS_SEVERELY_DELINQUENT", "sum"),
    CC_DPD_MAX=("SK_DPD", "max"),
    CC_DPD_DEF_MAX=("SK_DPD_DEF", "max"),
    CC_ACTIVE_MONTH_RATE=("CC_IS_ACTIVE", "mean"),
    CC_PAYMENT_BELOW_MINIMUM_RATE=("CC_PAYMENT_BELOW_MINIMUM", "mean"),
    CC_NEGATIVE_BALANCE_COUNT=("CC_NEGATIVE_BALANCE", "sum"),
    CC_NEGATIVE_RECEIVABLE_COUNT=("CC_NEGATIVE_RECEIVABLE", "sum"),
    CC_BALANCE_WITHOUT_LIMIT_COUNT=("CC_BALANCE_WITHOUT_LIMIT", "sum"),
    CC_PAYMENT_INFO_AVAILABLE_RATE=("CC_PAYMENT_INFO_AVAILABLE", "mean"),
    CC_DRAWING_INFO_AVAILABLE_RATE=("CC_DRAWING_INFO_AVAILABLE", "mean"),
    CC_RECORD_MISSING_RATE_MEAN=("CC_RECORD_MISSING_RATE", "mean"),
    CC_MATURE_INSTALLMENT_MAX=("CNT_INSTALMENT_MATURE_CUM", "max"),
).reset_index()
print("Credit-card accounts summarized:", len(account_features))
print("Account feature shape:", account_features.shape)

Credit-card accounts summarized: 87452
Account feature shape: (87452, 34)


Each credit-card account's monthly records are summarized here, average and maximum balance, utilization, and drawing/payment totals.


## Add recent 12-month account behaviour


In [16]:
recent_credit = credit.loc[credit["CC_RECENT_12M"].eq(1)]
recent_account = recent_credit.groupby(["SK_ID_CURR", "SK_ID_PREV"]).agg(
    CC_RECENT_12M_COUNT=("MONTHS_BALANCE", "count"),
    CC_RECENT_12M_BALANCE_MEAN=("AMT_BALANCE", "mean"),
    CC_RECENT_12M_UTILIZATION_MEAN=("CC_UTILIZATION", "mean"),
    CC_RECENT_12M_UTILIZATION_MAX=("CC_UTILIZATION", "max"),
    CC_RECENT_12M_DELINQUENT_COUNT=("CC_IS_DELINQUENT", "sum"),
    CC_RECENT_12M_DELINQUENT_RATE=("CC_IS_DELINQUENT", "mean"),
    CC_RECENT_12M_DPD_MAX=("SK_DPD", "max"),
    CC_RECENT_12M_PAYMENT_TOTAL=("AMT_PAYMENT_TOTAL_CURRENT", "sum"),
).reset_index()
account_features = account_features.merge(
    recent_account, on=["SK_ID_CURR", "SK_ID_PREV"], how="left", validate="one_to_one"
)
recent_columns = [c for c in recent_account.columns if c not in ["SK_ID_CURR", "SK_ID_PREV"]]
account_features[recent_columns] = account_features[recent_columns].fillna(0)
del recent_credit, recent_account, credit
print("Account feature shape after recent features:", account_features.shape)

Account feature shape after recent features: (87452, 42)


Same kind of summary as before, but limited to the most recent 12 months, to capture more current card behaviour.


## Aggregate credit-card accounts to applicant level


In [19]:
aggregation_rules = {}
for column in [c for c in account_features.columns if c not in ["SK_ID_CURR", "SK_ID_PREV"]]:
    if column.endswith("_COUNT") or column.endswith("_TOTAL"):
        aggregation_rules[column] = "sum"
    elif column == "CC_OLDEST_MONTH":
        aggregation_rules[column] = "min"
    elif column == "CC_LATEST_MONTH" or column.endswith("_MAX"):
        aggregation_rules[column] = "max"
    else:
        aggregation_rules[column] = "mean"

credit_features = account_features.groupby("SK_ID_CURR").agg(aggregation_rules).reset_index()
account_counts = account_features.groupby("SK_ID_CURR")["SK_ID_PREV"].nunique().rename("CC_ACCOUNT_COUNT").reset_index()
credit_features = credit_features.merge(account_counts, on="SK_ID_CURR", how="left", validate="one_to_one")
print("Applicants with card history summarized:", len(credit_features))
print("Applicant history feature shape:", credit_features.shape)

Applicants with card history summarized: 86905
Applicant history feature shape: (86905, 42)


Applicants with more than one credit card have their account-level summaries combined here into a single row per applicant.


## Add all applicants and encode structural absence


In [22]:
all_applicants = application_target[["SK_ID_CURR"]].copy()
all_applicants["CC_HISTORY_AVAILABLE"] = all_applicants["SK_ID_CURR"].isin(set(credit_features["SK_ID_CURR"])).astype("int8")
credit_features = all_applicants.merge(credit_features, on="SK_ID_CURR", how="left", validate="one_to_one")
feature_columns_before_rules = [c for c in credit_features.columns if c != "SK_ID_CURR"]
credit_features[feature_columns_before_rules] = credit_features[feature_columns_before_rules].fillna(0)
print("All applicants included:", len(credit_features))
print("Applicants with credit-card history:", int(credit_features["CC_HISTORY_AVAILABLE"].sum()))
print("Applicants without credit-card history:", int(credit_features["CC_HISTORY_AVAILABLE"].eq(0).sum()))

All applicants included: 307511
Applicants with credit-card history: 86905
Applicants without credit-card history: 220606


Applicants with no credit-card history at all are added back in here, with their feature values set to 0 rather than left missing, since having no card history is itself meaningful information, not something unknown.


## Apply training-only missingness and constant-feature rules


In [25]:
MISSING_THRESHOLD = 0.50
training_base = pd.DataFrame({"SK_ID_CURR": training_ids}).merge(
    application_target, on="SK_ID_CURR", how="left", validate="one_to_one"
).merge(credit_features, on="SK_ID_CURR", how="left", validate="one_to_one")
decision_rows = []
for feature in [c for c in credit_features.columns if c != "SK_ID_CURR"]:
    series = training_base[feature]
    missing_rate = series.isna().mean()
    unique_non_missing = series.nunique(dropna=True)
    correlation = series.corr(training_base["TARGET"]) if unique_non_missing > 1 else np.nan
    decision = "Keep"
    reason = "Retain for global cross-validated feature selection"
    if missing_rate >= MISSING_THRESHOLD:
        decision = "Remove"
        reason = f"Training-applicant missing rate is at least {MISSING_THRESHOLD:.0%}"
    elif unique_non_missing <= 1:
        decision = "Remove"
        reason = "Constant in the training set"
    decision_rows.append({
        "feature": feature, "missing_count": int(series.isna().sum()),
        "missing_rate": missing_rate, "unique_non_missing": int(unique_non_missing),
        "pearson_target_correlation": correlation,
        "absolute_correlation": abs(correlation) if pd.notna(correlation) else np.nan,
        "decision": decision, "reason": reason,
    })
feature_decisions = pd.DataFrame(decision_rows).sort_values(
    ["decision", "absolute_correlation"], ascending=[True, False]
).reset_index(drop=True)
removed_features = feature_decisions.loc[feature_decisions["decision"] == "Remove", "feature"].tolist()
credit_features = credit_features.drop(columns=removed_features)
print("Features removed:", removed_features)
print("Features retained:", credit_features.shape[1] - 1)
feature_decisions.round(5)

Features removed: []
Features retained: 42


,feature,missing_count,missing_rate,unique_non_missing,pearson_target_correlation,absolute_correlation,decision,reason
0,CC_RECENT_12M_UTILIZATION_MEAN,0,0.0,30102,0.07421,0.07421,Keep,Retain for global cross-validated feature sele...
1,CC_RECENT_12M_UTILIZATION_MAX,0,0.0,29657,0.07048,0.07048,Keep,Retain for global cross-validated feature sele...
2,CC_UTILIZATION_MEAN,0,0.0,47299,0.06249,0.06249,Keep,Retain for global cross-validated feature sele...
3,CC_RECENT_12M_BALANCE_MEAN,0,0.0,30331,0.04790,0.04790,Keep,Retain for global cross-validated feature sele...
4,CC_BALANCE_MEAN,0,0.0,47559,0.04780,0.04780,Keep,Retain for global cross-validated feature sele...
5,CC_UTILIZATION_MAX,0,0.0,45200,0.04252,0.04252,Keep,Retain for global cross-validated feature sele...
6,CC_MINIMUM_PAYMENT_MEAN,0,0.0,45925,0.04082,0.04082,Keep,Retain for global cross-validated feature sele...
7,CC_PAYMENT_BELOW_MINIMUM_RATE,0,0.0,768,0.03882,0.03882,Keep,Retain for global cross-validated feature sele...
8,CC_BALANCE_MAX,0,0.0,45597,0.03746,0.03746,Keep,Retain for global cross-validated feature sele...
9,CC_DRAWING_MEAN,0,0.0,39684,0.03551,0.03551,Keep,Retain for global cross-validated feature sele...


No credit-card features needed to be removed, all 42 stayed within the limits.


## Validate the applicant-level feature table


In [28]:
numeric_columns = credit_features.select_dtypes(include="number").columns
infinite_count = sum(int(np.isinf(credit_features[c].dropna()).sum()) for c in numeric_columns)
retained_decisions = feature_decisions.loc[feature_decisions["decision"] == "Keep"]
validation_checks = pd.DataFrame([
    {"check": "One row per project applicant", "passed": credit_features["SK_ID_CURR"].is_unique and len(credit_features) == len(application_target)},
    {"check": "All project applicants included", "passed": set(credit_features["SK_ID_CURR"]) == project_ids},
    {"check": "No TARGET in output", "passed": "TARGET" not in credit_features.columns},
    {"check": "No previous-account ID in output", "passed": "SK_ID_PREV" not in credit_features.columns},
    {"check": "History availability retained", "passed": "CC_HISTORY_AVAILABLE" in credit_features.columns},
    {"check": "No missing values after structural encoding", "passed": credit_features.isna().sum().sum() == 0},
    {"check": "No infinite numerical values", "passed": infinite_count == 0},
    {"check": "No retained feature reaches 50 percent training missingness", "passed": not retained_decisions["missing_rate"].ge(MISSING_THRESHOLD).any()},
    {"check": "Final test excluded from feature decisions", "passed": not training_base["SK_ID_CURR"].isin(test_id_set).any()},
])
assert validation_checks["passed"].all(), "At least one credit-card feature-engineering check failed."
validation_checks

,check,passed
0,One row per project applicant,True
1,All project applicants included,True
2,No TARGET in output,True
3,No previous-account ID in output,True
4,History availability retained,True
5,No missing values after structural encoding,True
6,No infinite numerical values,True
7,No retained feature reaches 50 percent trainin...,True
8,Final test excluded from feature decisions,True


All checks passed.


## Save features and audit reports


In [31]:
credit_features.to_pickle(output_path)
feature_decisions.to_csv(audit_folder / "credit_card_engineered_feature_decisions.csv", index=False)
validation_checks.to_csv(audit_folder / "credit_card_feature_engineering_validation.csv", index=False)
print("Credit-card feature table saved:", output_path)
print("Output rows:", len(credit_features))
print("Output columns:", credit_features.shape[1])
print("Applicants with history:", int(credit_features["CC_HISTORY_AVAILABLE"].sum()))
print("Remaining numerical missing values:", int(credit_features.select_dtypes(include="number").isna().sum().sum()))

Credit-card feature table saved: /Users/taranveersingh/A-MRP/data/features/credit_card_features.pkl
Output rows: 307511
Output columns: 43
Applicants with history: 86905
Remaining numerical missing values: 0


## Main feature engineering results

This notebook built 42 applicant-level features from the cleaned monthly credit-card data, covering balances, utilization, drawings, payments, and delinquency, plus a flag for whether the applicant has any card history at all.

All checks passed, and no features were removed. The feature table has 43 columns for all 307,511 applicants, with zero remaining missing values since applicants without credit-card history are filled with 0. The next step is to build features from the POS/cash data.
